In [ ]:
library(tidyr)
library(dplyr)
library(ggplot2)
library(patchwork)
library(data.table)

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/module_projection_fxns.R")

setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

options(repr.plot.width=20, repr.plot.height=8, repr.plot.res=150)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


Loading required package: future


Attaching package: ‘reshape2’


The following objects are masked from ‘package:data.table’:

    dcast, melt


The following object is masked from ‘package:tidyr’:

    smiths




Here I visualize module genes after "cleaning" the bulk data

In [3]:
mod_def <- "TopModPosBC"

### Bulk data and enrichments

In [4]:
# *** Bulk gene expression should be the same dataset used in to find modules and enrichments ***

bulk_data_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_42147genes_cleaned"
bulk_expr_file <- "data/cleaned/TopModPosFDR/GTEx_cortex_counts_TMMF_All_501_outliers_removed_42147genes_cleaned.csv"
bulk_expr <- fread(bulk_expr_file, data.table=FALSE)
colnames(bulk_expr)[1] <- "Gene"

enrichment_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_42147genes_cleaned_mergeParam0.95_subsetCutoff8_Modules_Claude_marker_genes_enrichments_top_Qval_mods"

top_mods_df <- fread(paste0("data/enrichments/", enrichment_source, ".csv"), data.table=FALSE)

max_qval <- 1 

top_mods_df <- top_mods_df[!is.na(top_mods_df$Qval),]
top_mods_df_subset <- top_mods_df[top_mods_df$Qval < max_qval,]

dim(top_mods_df_subset)

[1] 27  7

### Prep single cell data

In [ ]:
# sc_data_source <- "yao_2021_MOp_STAR_gene_counts_normalized"

# sc_expr <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/MOp/yao_2021_MOp_STAR_gene_counts.csv", data.table=FALSE)
# sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/MOp/yao_2021_MOp_STAR_sampleinfo.csv", data.table=FALSE)
# colnames(sc_expr)[1] <- "Gene"

# # Remove cell types with only a few cells

# min_cells <- 5

# ctype_tally <- table(sampleinfo$subclass_label)
# ctypes_to_keep <- names(ctype_tally)[ctype_tally >= min_cells]
# cells_to_keep <- which(sampleinfo$subclass_label %in% ctypes_to_keep)

# sc_expr <- sc_expr[, c(1, cells_to_keep + 1)]
# sampleinfo <- sampleinfo[cells_to_keep,]

# all.equal(colnames(sc_expr)[-1], sampleinfo$Cell_ID)

# total_expr <- colSums(sc_expr[,-1])
# sc_expr_norm <- sweep(sc_expr[,-1], MARGIN=2, FUN="/", STATS=total_expr) * 1e4
# sc_expr_norm <- data.frame(Gene=sc_expr[,1], sc_expr_norm) 

# ctype_assignment_vec <- sampleinfo$subclass_label

[1] TRUE

In [ ]:
sc_data_source <- "yao_2021_ACA_STAR_gene_counts_normalized"

sc_expr <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/ACA/yao_2021_ACA_STAR_gene_counts.csv", data.table=FALSE)
sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/ACA/yao_2021_ACA_STAR_sampleinfo.csv", data.table=FALSE)
colnames(sc_expr)[1] <- "Gene"

# Remove cell types with only a few cells

min_cells <- 5

ctype_tally <- table(sampleinfo$subclass_label)
ctypes_to_keep <- names(ctype_tally)[ctype_tally >= min_cells]
cells_to_keep <- which(sampleinfo$subclass_label %in% ctypes_to_keep)

sc_expr <- sc_expr[, c(1, cells_to_keep + 1)]
sampleinfo <- sampleinfo[cells_to_keep,]

all.equal(colnames(sc_expr)[-1], sampleinfo$Cell_ID)

total_expr <- colSums(sc_expr[,-1])
sc_expr_norm <- sweep(sc_expr[,-1], MARGIN=2, FUN="/", STATS=total_expr) * 1e4
sc_expr_norm <- data.frame(Gene=sc_expr[,1], sc_expr_norm) 

ctype_assignment_vec <- sampleinfo$subclass_label

[1] TRUE

## Visualize module genes

In [13]:
outdir <- paste0("figures/module_genes/", sc_data_source, "/", mod_def)

if (!dir.exists(outdir)) {
    dir.create(outdir, recursive=TRUE)
}

filename <- paste0(outdir, "/", enrichment_source, ".pdf")

max_genes <- 15

pdf(file=filename)

for (i in 1:nrow(top_mods_df_subset)) {
    print(paste("Module", i))

    mod <- top_mods_df_subset$Module[i]
    kME_path <- top_mods_df_subset$kME_path[i]
    mod_genes <- get_mod_genes(kME_path, mod, mod_def)

    plot_title <- paste(
        top_mods_df_subset$Cell_type[i], mod_def, 
        "\n", mod, top_mods_df_subset$Network[i]
    ) 
    plot_sub <- paste("Qval:", round(top_mods_df_subset$Qval[i], 4))

    # Plot gene expression over bulk samples
    
    mod_genes_subset <- na.omit(mod_genes[1:max_genes])
    plot_gene_expr_over_samples(bulk_expr, mod_genes_subset, plot_title, plot_sub, target_species=NULL)

    # Plot module genes in single cell data

    plot_gene_projections(sc_expr_norm, mod_genes, ctype_assignment_vec, plot_title, plot_sub, target_species="mouse")
}

dev.off()

[1] "Module 1"
[1] "Module 2"
[1] "Module 3"
[1] "Module 4"
[1] "Module 5"
[1] "Module 6"
[1] "Module 7"
[1] "Module 8"
[1] "Module 9"
[1] "Module 10"
[1] "Module 11"
[1] "Module 12"
[1] "Module 13"
[1] "Module 14"
[1] "Module 15"
[1] "Module 16"
[1] "Module 17"
[1] "Module 18"
[1] "Module 19"
[1] "Module 20"
[1] "Module 21"
[1] "Module 22"
[1] "Module 23"
[1] "Module 24"
[1] "Module 25"
[1] "Module 26"
[1] "Module 27"


agg_record_200157652 
                   2

In [18]:
dev.off()

pdf 
  3

## Looking more closely at modules of interest

In [ ]:
top_mods_df_subset <- top_mods_df[top_mods_df$Cell_type == "Sncg",]

In [ ]:
enrichments <- fread("data/enrichments/GTEx_cortex_counts_TMMF_All_501_outliers_removed_42147genes_cleaned_mergeParam0.95_subsetCutoff8_Modules_MO_2117sets_enrichments.csv", data.table=FALSE)

In [ ]:
mask <- grepl(top_mods_df_subset$Network, enrichments$Network) & grepl(paste0("^", top_mods_df_subset$Module, "$"), enrichments$Module) 
enrichments_subset <- enrichments[mask,]

print(enrichments_subset %>% filter(Qval < .05))

,Network,kME_path,ME_path,Pval,Cell_type,Module,Qval
,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>
17,Bicor-None_signum0.663_minSize10_merge_ME_0.95_16795,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_counts_TMMF_All_501_outliers_removed_42147genes_cleaned_mergeParam0.95_subsetCutoff8_Modules/Bicor-None_signum0.663_minSize10_merge_ME_0.95_16795/kME_table_01-18-05.csv,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_counts_TMMF_All_501_outliers_removed_42147genes_cleaned_mergeParam0.95_subsetCutoff8_Modules/Bicor-None_signum0.663_minSize10_merge_ME_0.95_16795/Module_eigengenes_01-18-05.csv,9.876062e-05,Sncg,tomato,0.05833353


In [ ]:
enrichments <- fread("data/enrichments/GTEx_cortex_counts_TMMF_All_501_outliers_removed_42147genes_cleaned_mergeParam0.95_subsetCutoff8_Module", data.table=FALSE)

In [ ]:
mask <- grepl(top_mods_df_subset$Network, enrichments$Network) & grepl(paste0("^", top_mods_df_subset$Module, "$"), enrichments$Module) 
enrichments_subset <- enrichments[mask,]

print(enrichments_subset %>% filter(Qval < .05))